<a href="https://colab.research.google.com/github/qubit55/clojupyter-playground/blob/main/bpe-clojure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Clojupyter kernel
!wget https://raw.githubusercontent.com/qubit55/clojupyter_colab_setup/refs/heads/main/install_clojure_kernel.sh
!chmod +x install_clojure_kernel.sh
!./install_clojure_kernel.sh

In [10]:
(require '[clojupyter.misc.helper :as helper]
         '[clojupyter.display :as display])

(helper/add-dependencies '[com.knuddels/jtokkit "1.1.0"])
(helper/add-dependencies '[criterium "0.4.6"])

(use 'criterium.core)

nil

In [41]:
(import '[com.knuddels.jtokkit Encodings]
        '[com.knuddels.jtokkit.api EncodingType])

(def encoder
 (.getEncoding (Encodings/newDefaultEncodingRegistry)
               EncodingType/R50K_BASE))

(def jtok-encodings (.encode encoder text))

#'user/jtok-encodings

In [42]:
(def text
 (slurp "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"))

#'user/text

In [43]:
(def jtok-encodings (.encode encoder text))

#'user/jtok-encodings

In [44]:
(type jtok-encodings)

class com.knuddels.jtokkit.api.IntArrayList

In [45]:
(count (.toArray jtok-encodings))

5145

In [54]:
(bench (.encode encoder text))

Evaluation count : 34020 in 60 samples of 567 calls.
             Execution time mean : 2.016093 ms
    Execution time std-deviation : 457.308118 µs
   Execution time lower quantile : 1.753224 ms ( 2.5%)
   Execution time upper quantile : 3.192285 ms (97.5%)
                   Overhead used : 12.027816 ns

Found 13 outliers in 60 samples (21.6667 %)
	low-severe	 1 (1.6667 %)
	low-mild	 12 (20.0000 %)
 Variance from outliers : 92.8924 % Variance is severely inflated by outliers


nil

In [50]:
;; (def tik-encodings
;;   (map Integer/parseInt (clojure.string/split (slurp "/content/token_ids.txt") #"\n")))

#'user/tik-encodings

In [53]:
;; (count (filter false? (map = tik-encodings (seq (.toArray jtok-encodings)))))

0

In [ ]:
(quick-bench (encode text))

In [ ]:
(quick-bench (encode-2 text))

In [ ]:
(quick-bench (encode-2-1 text))

In [ ]:
(with-progress-reporting (quick-bench (encode-3 text)))

In [ ]:
(count (encode-2 text))

In [ ]:
(count text)

In [ ]:
(count (encode-2 text))

In [ ]:
(take-last 1 (encode-2 text))

In [ ]:
(import [java.util.regex Pattern Matcher])

In [ ]:
(defn find-all [regex text]
  (let [pattern (Pattern/compile regex)
        matcher (.matcher pattern text)
        tokens  (loop [matches []]
                  (if (.find matcher)
                      (recur (conj matches (.group matcher)))
                  matches))]
   tokens))

(defn find-all [regex text]
  (let [pattern (Pattern/compile regex)
        matcher (.matcher pattern text)
        tokens  (loop [matches []]
                  (if (.find matcher)
                    (recur (conj matches (.group matcher)))
                    matches))]
    (map #(clojure.string/replace % #"^ " "Ġ") tokens)))  ;; Replace leading space with "Ġ"

In [ ]:
(def r50k_pat_str "'s|'t|'re|'ve|'m|'ll|'d| ?[\\p{L}]+| ?[\\p{N}]+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+")

In [ ]:
(def test-text "Hello, world. Is this-- a test?\nA new test?")

In [ ]:
(def test-text-tokens (find-all r50k_pat_str test-text))

In [ ]:
test-text-tokens

In [ ]:
(count test-text-tokens)

In [ ]:
(count (find-all r50k_pat_str text))

In [ ]:
(quick-bench (find-all r50k_pat_str text))

In [ ]:
(type (slurp "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe"))

In [ ]:
(def vocab-url "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/encoder.json")

In [ ]:
(def merges-url "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe")

In [ ]:
(def inverse-vocab (json/read-str (slurp vocab-url)))
(def vocab (clojure.set/map-invert inverse-vocab))

In [ ]:
(inverse-vocab "ĠNewman")

In [ ]:
(vocab 27698)

In [ ]:
(take 3 vocab)

In [ ]:
(map (fn [c] (inverse-vocab (str c))) "ĠHello")

In [ ]:
(def bpe-merges (slurp merges-url))

In [ ]:
(defn bpe-merges-to-id
 [merges inverse-vocab]
 (->>
  (clojure.string/split merges #"\n")
  (drop-while #(clojure.string/starts-with? % "#"))  ;; Remove only the first header if it starts with #
  (map (fn [line] (clojure.string/split (clojure.string/trim line) #"\s+")))
  (map (fn [[tk-1 tk-2]] {[(inverse-vocab tk-1) (inverse-vocab tk-2)] (inverse-vocab (str tk-1 tk-2))}))
  (apply merge)
  ))

In [ ]:
(->> (bpe-merges-to-id "#acvd\nĠreg ress
ĠColl ider
Ġinform ants
Ġg azed" inverse-vocab)
)

In [ ]:
(def bpe-table (bpe-merges-to-id bpe-merges inverse-vocab))

In [ ]:
(def inverse-bpe-table (clojure.set/map-invert bpe-table))

In [ ]:
(take 3 bpe-table)

In [ ]:
(bpe-table [42 78])

In [ ]:
(def not-nil? (complement nil?))

In [ ]:
(defn tokenize-with-bpe
 [token inverse-vocab bpe-id-table]
 (->> token
  (map (fn [c] (inverse-vocab (str c))))
  (partition 2 2 nil)
  (map (fn [[idl idr]]
   (if (not-nil? idr)
    (if (not-nil? (bpe-id-table [idl idr]))
     (bpe-id-table [idl idr])
     idl)
    idl
    ))
   )
  )
)

In [ ]:
(tokenize-with-bpe "ĠHello" inverse-vocab bpe-table)

In [ ]:
(defn encode
 [text pat inverse-vocab bpe-id-table]
 (->> (find-all pat text)
   (map (fn [token]
         (if (not-nil? (inverse-vocab token))
          (inverse-vocab token)
          (tokenize-with-bpe token inverse-vocab bpe-id-table)
          )))
   (flatten)))

In [ ]:
(inverse-vocab "ĠHello")

In [ ]:
(find-all r50k_pat_str "ĠHello")

In [ ]:
(inverse-vocab )

In [ ]:
(encode "ĠHello" r50k_pat_str inverse-vocab bpe-table)

In [ ]:
[128, 254, 28254, 2238, 198, 128, 254, 28254, 2238]

In [ ]:
(encode "Hello, world. Is this-- a test? A new test?"
        r50k_pat_str inverse-vocab bpe-table)

In [ ]:
text

In [ ]:
[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30, 317, 649, 1332, 30]

In [ ]:
(def toked-text
 (encode text r50k_pat_str inverse-vocab bpe-table))

In [ ]:
;;(I ĠHAD Ġalways Ġthought ĠJack ĠGisburn Ġrather Ġa Ġcheap Ġgenius)

In [ ]:
toked-text

In [ ]:
[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]

In [ ]:
;; 11110, 35906

In [ ]:
(take 10 (find-all r50k_pat_str text))

In [ ]:
(def bpe-table-inv (clojure.set/map-invert bpe-table))

In [ ]:
(bpe-table-inv 10899)

In [ ]:
(vocab 65)

In [ ]:
(vocab 700)

In [ ]:
"I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow "

In [ ]:
(def my-encoding
 (encode "I HAD always thought Jack Gisburn"
  r50k_pat_str inverse-vocab bpe-table))
my-encoding

In [ ]:
(def tiktoken-encoding
 [40,
 367,
 2885,
 1464,
 1807,
 3619,
 402,
 271,
 10899,
 2138,
 257,
 7026,
 15632,
 438,
 2016,
 257,
 922,
 5891,
 220])

In [ ]:
(map = my-encoding tiktoken-encoding)